# 01 - Predictive Model Evaluation

**Objective.** Verify the tracked research experiment registry, distinguish the research winner from the deployed transferable artifact, check deployed-model hashes, and optionally recompute predictive metrics from external prediction rows. No model is trained or substituted.

**Inputs.** The actual repository backend is resolved relative to this notebook. Large prediction Parquet files are optional and supplied through `C2_HISTORICAL_ROOT`.

In [1]:
from pathlib import Path
import os
import pandas as pd
from evaluation_runner import evaluate_predictive

HERE = Path.cwd().resolve()
assert (HERE / 'evaluation_runner.py').is_file(), 'Run the notebook from the evaluation folder.'
SOURCE_BACKEND = HERE.parents[2]
external = os.environ.get('C2_HISTORICAL_ROOT')
HISTORICAL_ROOT = Path(external).resolve() if external else None
print('Repository-relative backend:', SOURCE_BACKEND.relative_to(SOURCE_BACKEND.parent))
print('External prediction evidence supplied:', bool(HISTORICAL_ROOT))

Repository-relative backend: backend
External prediction evidence supplied: True


In [2]:
result = evaluate_predictive(SOURCE_BACKEND, HISTORICAL_ROOT, bootstrap_replicates=20)
model_rows = [
    {'artifact': result['research_winner_frozen']['model_version'], 'role': 'research winner', **{k: result['research_winner_frozen'][k] for k in ['roc_auc','average_precision','log_loss','brier_score','ece_10_equal_frequency']}},
    {'artifact': result['deployed_transferable_frozen']['model_version'], 'role': 'deployed transferable', 'roc_auc': result['deployed_transferable_frozen']['roc_auc'], 'average_precision': None, 'log_loss': result['deployed_transferable_frozen']['log_loss'], 'brier_score': result['deployed_transferable_frozen']['brier_score'], 'ece_10_equal_frequency': None},
]
pd.DataFrame(model_rows)

,artifact,role,roc_auc,average_precision,log_loss,brier_score,ece_10_equal_frequency
0,H2_GB_RT_OFF_v1,research winner,0.750158,0.851203,0.534442,0.178955,0.003847
1,TRANSFER_H2_GB_RT_OFF_v1,deployed transferable,0.712077,NaN,0.563200,0.190298,NaN


In [3]:
integrity = pd.DataFrame(result['artifact_identity']['manifest_integrity']).T
display(integrity[['exists','bytes','sha256','matches_manifest']])
print('Eight registered experiments:', result['eight_model_registry']['unique_experiments'])
print('All deployed manifest files match:', result['artifact_identity']['all_manifest_files_match'])

,exists,bytes,sha256,matches_manifest
model.json,True,1233236,dc31acb1307568b9f7886d98cf163650ba2eb86ed59c93...,True
calibrator.joblib,True,831,b4d889b0cd99b572e0ed7ba25fde1cc05e8522994f69a1...,True
metadata.json,True,2053,c55f1049eb3e27dfffca169ef0ff6cede5ea0565d1c452...,True
evaluation.json,True,1062,2bc6a95291a2ec1a6984524a8ef36a508382dc4832f3d7...,True


Eight registered experiments: 8
All deployed manifest files match: True


In [4]:
recomputed = result['prediction_level_recomputation']
if recomputed.get('available'):
    core = recomputed['metrics']
    display(pd.DataFrame([{k: core[k] for k in ['n_interactions','roc_auc','average_precision','accuracy','f1','log_loss','brier_score','ece_10_equal_frequency']}]))
    display(pd.DataFrame(recomputed['history_length_subgroups']))
    display(pd.DataFrame(core['calibration_bins']))
else:
    print('Metadata-only verification path:', recomputed['reason'])
    print('Set C2_HISTORICAL_ROOT to recompute Accuracy, F1, calibration bins and clustered intervals.')

,n_interactions,roc_auc,average_precision,accuracy,f1,log_loss,brier_score,ece_10_equal_frequency
0,937043,0.750158,0.851203,0.734105,0.821907,0.534442,0.178955,0.003847


,history_length,n_interactions,positive_prevalence,roc_auc,average_precision,log_loss,brier_score,accuracy
0,first,6981,0.665664,0.646748,0.764677,0.605439,0.208211,0.686005
1,2-4,20253,0.671703,0.724870,0.832769,0.559618,0.189190,0.714906
2,5-10,37076,0.659996,0.731778,0.829177,0.560399,0.189641,0.710541
3,11+,872733,0.677237,0.752154,0.852918,0.532187,0.178029,0.735936


,bin,n,mean_predicted,observed_fraction,absolute_gap
0,1,93705,0.246114,0.249058,0.002944
1,2,93705,0.456314,0.464308,0.007994
2,3,93705,0.564539,0.562297,0.002242
3,4,93704,0.639667,0.636088,0.003579
4,5,93704,0.698418,0.697878,0.000539
5,6,93704,0.747110,0.741132,0.005978
6,7,93704,0.789679,0.786829,0.002850
7,8,93704,0.829863,0.823113,0.006750
8,9,93704,0.871747,0.870560,0.001186
9,10,93704,0.927828,0.932233,0.004406


## Interpretation and limitations

ROC-AUC and probability metrics describe source-domain next-response prediction. They are not recommendation-effectiveness, local-child calibration, clinical-validity, or learning-improvement measures. `H2_GB_RT_OFF_v1` and `TRANSFER_H2_GB_RT_OFF_v1` are distinct trained artifacts.